<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>📈 Chapter 7 — Supervised Learning: Regression</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 75 mins | Level: Intermediate</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Explain and implement simple and multiple linear regression
- Understand regression metrics: MSE, RMSE, MAE, R²
- Use sklearn's LinearRegression, Ridge, and Lasso
- Understand overfitting and regularisation (L1 / L2)
- Build a complete salary prediction pipeline with sklearn

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model  import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler
from sklearn.metrics        import mean_squared_error, mean_absolute_error, r2_score

%matplotlib inline
np.random.seed(42)

# ── Dataset: Indian tech employees ────────────────────────────
n = 500
df = pd.DataFrame({
    'experience':   np.random.randint(0, 20, n),
    'education':    np.random.choice([0, 1, 2], n),
    'city_tier':    np.random.choice([1, 2, 3], n),
    'skills_score': np.random.randint(40, 100, n),
    'dept':         np.random.choice(['Eng', 'Sales', 'HR', 'Finance'], n),
})
df['salary'] = (
    300000
    + df['experience']  * 80000
    + df['education']   * 120000
    + (4 - df['city_tier']) * 50000
    + df['skills_score'] * 2000
    + np.random.normal(0, 80000, n)
).astype(int)

df = pd.get_dummies(df, columns=['dept'], drop_first=True)
print('✅ Dataset shape:', df.shape)
df.head()

## 📖 Section A — Regression Fundamentals

**Linear Regression** fits: $\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n$

It minimises MSE: $\text{MSE} = \frac{1}{n} \sum (y_i - \hat{y}_i)^2$

**Key Metrics:**

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| MSE | $\frac{1}{n}\sum(y-\hat{y})^2$ | Average squared error — penalises large errors |
| RMSE | $\sqrt{\text{MSE}}$ | Same units as target — easy to interpret |
| MAE | $\frac{1}{n}\sum|y-\hat{y}|$ | Robust to outliers |
| R² | $1 - \frac{SS_{res}}{SS_{tot}}$ | % of variance explained (1.0 = perfect) |

> 💡 **Key Rule:** R² = 0.85 means the model explains 85% of the variance in y. The remaining 15% is unexplained noise. R² can be negative if the model is worse than predicting the mean.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Linear Regression with sklearn
# ─────────────────────────────────────────────────────────────
feature_cols = [c for c in df.columns if c != 'salary']
X = df[feature_cols].values
y = df['salary'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)  # use SAME scaler — fit only on train!

model = LinearRegression()
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f'RMSE : ₹{rmse:,.0f}')
print(f'MAE  : ₹{mae:,.0f}')
print(f'R²   : {r2:.4f}')
print(f'\nTop coefficients:')
coef_df = pd.DataFrame({'feature': feature_cols, 'coef': model.coef_}).sort_values('coef', key=abs, ascending=False)
print(coef_df.head())

## 📖 Section B — Regularisation: Ridge and Lasso

**Regularisation** adds a penalty for large weights to prevent overfitting:

$$\text{Ridge (L2):} \quad \text{Loss} + \alpha \sum w_i^2 \quad \text{(shrinks weights, keeps all)}$$

$$\text{Lasso (L1):} \quad \text{Loss} + \alpha \sum |w_i| \quad \text{(can zero out weights = feature selection)}$$

| | Ridge | Lasso |
|---|---|---|
| Penalty | L2 (squared) | L1 (absolute) |
| Effect | Shrinks all weights | Zeroes out some weights |
| Good for | Collinear features | Feature selection |

> 💡 **Key Rule:** Start with Ridge. Use Lasso when you have many features and suspect only a few matter.

---
## 🟢 Exercise 7.1 — Compute Regression Metrics from Scratch *(Level 1 · Est. 10 min)*

> Implement `compute_metrics(y_true, y_pred)` returning a dict with MSE, RMSE, MAE, R². Verify against sklearn.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 7.1: Regression Metrics from Scratch
# ─────────────────────────────────────────────────────────────

def compute_metrics(y_true, y_pred):
    # YOUR CODE HERE
    pass

metrics = compute_metrics(y_test, y_pred)
print('My metrics:   ', {k: f'{v:.2f}' for k, v in metrics.items()})
print('Sklearn RMSE: ', f'{np.sqrt(mean_squared_error(y_test, y_pred)):.2f}')
print('Sklearn R²:   ', f'{r2_score(y_test, y_pred):.4f}')

assert abs(metrics['RMSE'] - np.sqrt(mean_squared_error(y_test, y_pred))) < 1
assert abs(metrics['R2']   - r2_score(y_test, y_pred))                  < 0.001
print('\n✅ Metrics verified!')

In [ ]:
# @title ✅ Solution — Exercise 7.1 (click to expand)

def compute_metrics(y_true, y_pred):
    mse    = np.mean((y_true - y_pred) ** 2)
    rmse   = np.sqrt(mse)
    mae    = np.mean(np.abs(y_true - y_pred))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2     = 1 - ss_res / ss_tot
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

metrics = compute_metrics(y_test, y_pred)
print('Metrics:', {k: round(v, 2) for k, v in metrics.items()})
assert abs(metrics['R2'] - r2_score(y_test, y_pred)) < 0.001
print('✅ Solution verified!')

---
## 🟡 Exercise 7.2 — Compare LinearRegression, Ridge, Lasso *(Level 2 · Est. 15 min)*

> Train all three models. Compare R² and RMSE. Plot coefficient values side by side.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 7.2: Model Comparison
# ─────────────────────────────────────────────────────────────

models = {
    'LinearRegression': LinearRegression(),
    'Ridge(alpha=100)':  Ridge(alpha=100),
    'Lasso(alpha=1000)': Lasso(alpha=1000),
}

results = {}
# YOUR CODE HERE — train each model, compute RMSE and R², store in results

print('Model Comparison:')
for name, res in results.items():
    print(f'{name:25s}  RMSE: ₹{res["RMSE"]:>12,.0f}  R²: {res["R2"]:.4f}')

# Plot coefficients side by side
# YOUR CODE HERE

In [ ]:
# @title ✅ Solution — Exercise 7.2 (click to expand)

models = {
    'LinearRegression': LinearRegression(),
    'Ridge(100)':       Ridge(alpha=100),
    'Lasso(1000)':      Lasso(alpha=1000),
}

results = {}
coefs   = {}
for name, m in models.items():
    m.fit(X_train_s, y_train)
    yp = m.predict(X_test_s)
    results[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, yp)), 'R2': r2_score(y_test, yp)}
    coefs[name]   = m.coef_

print('Model Comparison:')
for name, res in results.items():
    print(f'{name:22s}  RMSE: ₹{res["RMSE"]:>12,.0f}  R²: {res["R2"]:.4f}')

# Coefficient comparison
x = np.arange(len(feature_cols))
width = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, c) in enumerate(coefs.items()):
    ax.bar(x + i * width, c, width, label=name)
ax.set_xticks(x + width)
ax.set_xticklabels(feature_cols, rotation=30, ha='right')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Coefficients: Linear vs Ridge vs Lasso')
ax.legend()
plt.tight_layout()
plt.show()
print('\n✅ Notice Lasso zeros out some coefficients — automatic feature selection!')

---
## 🔴 Exercise 7.3 — Polynomial Features + Overfitting Demo *(Level 3 · Est. 25 min)*

> Using a 1D dataset, fit polynomial regression for degrees 1–10. Plot train vs test RMSE. Show overfitting.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 7.3: Polynomial Regression and Overfitting
# ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline       import Pipeline

# 1D data: y = sin(x) + noise
np.random.seed(42)
x_1d     = np.linspace(0, 2*np.pi, 60)
y_1d     = np.sin(x_1d) + np.random.normal(0, 0.3, 60)
X_1d     = x_1d.reshape(-1, 1)

train_rmse, test_rmse = [], []
degrees = range(1, 11)

for deg in degrees:
    # Build pipeline: PolynomialFeatures → StandardScaler → LinearRegression
    # YOUR CODE HERE
    # Compute train and test RMSE
    pass

plt.figure(figsize=(9, 5))
plt.plot(degrees, train_rmse, 'bo-', label='Train RMSE')
plt.plot(degrees, test_rmse,  'ro-', label='Test RMSE')
plt.xlabel('Polynomial Degree')
plt.ylabel('RMSE')
plt.title('Overfitting: Train RMSE ↓ but Test RMSE ↑ beyond degree ~3')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# @title ✅ Solution — Exercise 7.3 (click to expand)
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline       import Pipeline

np.random.seed(42)
x_1d = np.linspace(0, 2*np.pi, 60)
y_1d = np.sin(x_1d) + np.random.normal(0, 0.3, 60)
X_1d = x_1d.reshape(-1, 1)

X_tr, X_te, y_tr, y_te = train_test_split(X_1d, y_1d, test_size=0.3, random_state=42)
train_rmse, test_rmse = [], []

for deg in range(1, 11):
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=deg, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model',  LinearRegression()),
    ])
    pipe.fit(X_tr, y_tr)
    train_rmse.append(np.sqrt(mean_squared_error(y_tr, pipe.predict(X_tr))))
    test_rmse.append( np.sqrt(mean_squared_error(y_te, pipe.predict(X_te))))

plt.figure(figsize=(9, 5))
plt.plot(range(1,11), train_rmse, 'bo-', label='Train RMSE')
plt.plot(range(1,11), test_rmse,  'ro-', label='Test RMSE')
plt.xlabel('Polynomial Degree')
plt.ylabel('RMSE')
plt.title('The Bias-Variance Trade-off')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print('\n✅ Classic overfitting signature: train RMSE keeps falling, test RMSE rises after degree ~3.')
print('The sweet spot (lowest test RMSE) is the optimal model complexity.')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: What is the difference between R² and RMSE?</strong></summary>

**Answer:** R² is a relative measure (0 to 1) — it tells you what fraction of variance is explained. It's scale-free, making it useful for comparing models across different targets. RMSE is an absolute measure in the same units as the target (e.g., ₹) — it tells you the average prediction error in real terms. Use R² to compare models; use RMSE to interpret business impact ('our model is off by ₹80,000 on average').
</details>

<details>
<summary><strong>Q2: What is overfitting? How do you detect it?</strong></summary>

**Answer:** Overfitting is when a model memorises training data and fails to generalise to unseen data. Detection: train RMSE is much lower than test RMSE. Root cause: model is too complex relative to the data size. Solutions: (1) add regularisation (Ridge/Lasso), (2) reduce model complexity, (3) collect more data, (4) use cross-validation to estimate true generalisation error.
</details>

<details>
<summary><strong>Q3: Why should you fit the scaler only on training data?</strong></summary>

**Answer:** Fitting the scaler on test data would cause data leakage — the model would indirectly 'see' the test set's distribution during training. Always call `scaler.fit_transform(X_train)` on the training set, then `scaler.transform(X_test)` on the test set using the SAME scaler fitted on training data. This simulates real deployment where you scale new data using statistics computed from training data.
</details>

<details>
<summary><strong>Q4: What is the difference between Ridge and Lasso regularisation?</strong></summary>

**Answer:** Ridge (L2) adds α∑w² to the loss — it shrinks all weights towards zero but rarely reaches exactly zero. Lasso (L1) adds α∑|w| — due to the sharp corner of the absolute value at zero, it forces some weights to exactly zero (sparse solution). This makes Lasso a built-in feature selector. Ridge is generally preferred when most features are relevant; Lasso when you suspect many features are irrelevant.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 7 Complete!</h3>
<ul>
<li>Linear regression: theory, sklearn, and from scratch</li>
<li>Regression metrics: MSE, RMSE, MAE, R²</li>
<li>Ridge and Lasso regularisation</li>
<li>Polynomial features and the overfitting/underfitting trade-off</li>
</ul>
<p><strong>Next:</strong> Chapter 8 — Supervised Learning: Classification</p>
</div>